# The Full Campaign Lifecycle: Submission to Calendar

This notebook covers the v2.2 milestone (Phases 26-29) end to end -- a campaign
(`tom_targets.TargetList`) and four `CampaignRun`s, taken all the way from public
submission to calendar events -- by driving every state transition through the REAL
staff-facing views with `django.test.Client`. Every write below goes through the same
`campaigns:submit`, `campaigns:decide` (`approve`/`resolve_site`) and
`campaigns:attribution_decide` (`confirm`) views a real submitter and a real staff member
would use.

This deliberately contrasts with `reconcile_campaign_runs_demo.ipynb`, which covers the
reconciler alone (`solsys_code/campaign_reconciler.py`), seeding its four `CampaignRun`s
directly against already-approved rows via `CampaignRun.objects.update_or_create(...,
approval_status=APPROVED)`. That is the right scope for demonstrating the reconciler in
isolation, but it skips every human workflow step the milestone actually added -- the
public intake form, the approval decision, site disambiguation, and operator-assisted
attribution. This notebook is the missing counterpart: it demonstrates those human
workflow steps, and only reaches the reconciler as their consequence.

This notebook lives in `pre_executed/` because it is **DB-dependent** (it seeds
`Observatory`/`TargetList` records and creates `CampaignRun`/`CalendarEvent` rows) and is
therefore **NOT** run during Sphinx/CI/ReadTheDocs builds, per `docs/notebooks/README.md`.

This notebook also covers the campaign-annotation half of the v2.2/v2.4 story (Phase 33):
`CalendarEventMeta.run` means the event is ATTRIBUTED to that run, never that the run owns
it. The decoration a visitor sees -- the "Attributed campaign run" block in the event modal
and the `.cal-campaign-chip` marker on the month calendar -- is rendered from that link at
request time, so it survives a from-scratch rewrite of the event's own title and description,
and disappears -- with nothing deleted -- the moment the link is cleared. And since Phase 33,
a reconciler-created event's title no longer carries the campaign name at all: the pop-up
renders it from the link instead.


## Django setup, and why this notebook calls `setup_test_environment()`

Standard boilerplate to make `src.fomo.settings` importable from this notebook's location
(`docs/notebooks/pre_executed/` -- three levels under the repo root, so `parents[2]` gives
the repo root) and to allow synchronous ORM calls inside Jupyter's async event loop, exactly
like `reconcile_campaign_runs_demo.ipynb`'s own setup cell.

This notebook additionally calls `django.test.utils.setup_test_environment()` once, guarded
in a `try`/`except RuntimeError` so a re-run of this notebook in the same kernel session
does not fail on the already-called guard. That call does two things, both required below:

1. It appends `testserver` -- `django.test.Client`'s default `SERVER_NAME` -- to
   `ALLOWED_HOSTS`. Without it, a bare `Client()` request raises `DisallowedHost`: this
   developer's `src/fomo/local_settings.py` (gitignored) sets a non-empty `ALLOWED_HOSTS`,
   and a developer without that file gets `ALLOWED_HOSTS = []` with `DEBUG = True`, which
   permits only localhost -- `testserver` is neither.
2. It swaps in Django's locmem email backend, so `CampaignRunSubmissionView._notify_staff()`'s
   staff-notification email is captured in `django.core.mail.outbox` below instead of being
   dumped to this notebook's console output.

**There is no separate test database here** in the Django-test-runner sense --
`django.test.Client` drives the real request/response cycle straight through the
ordinary views. But this notebook's setup cell below copies the developer database
(`src/fomo_db.sqlite3`) to a throwaway scratch file and points `FOMO_DATABASE_PATH` at
that copy before `django.setup()` runs, so every one of those requests writes to the
scratch copy, never to the developer database itself (UAT G-33-4). That is exactly why
every write below is ALSO demo-scoped: the reset cell further down deletes only this demo
campaign's own rows, by exact name, before creating anything -- re-runnability inside the
scratch copy, not protection of a shared database.

In [1]:
import os
import sys
from pathlib import Path

import django
import django.test.utils

# Ensure the repo root is on sys.path so `src.fomo.settings` is importable
# when this notebook is executed from docs/notebooks/pre_executed/.
# NOTE: parents[2] is correct only when the Jupyter kernel CWD is
# docs/notebooks/pre_executed/. Start Jupyter from that directory, or
# adjust the index if you launch from the repo root.
repo_root_path = Path.cwd().resolve().parents[2]
if not (repo_root_path / 'manage.py').exists():
    raise RuntimeError(f'No manage.py at {repo_root_path}; run Jupyter from docs/notebooks/pre_executed/')
repo_root = str(repo_root_path)
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'src.fomo.settings')

# Jupyter's ipykernel runs inside an asyncio event loop, but Django's ORM is
# sync-only by default and refuses to run there; this opts back in.
os.environ.setdefault('DJANGO_ALLOW_ASYNC_UNSAFE', 'true')

# Copy the developer database to a scratch file and point FOMO_DATABASE_PATH at the
# copy BEFORE django.setup() -- that call is what materialises DATABASES, so the
# variable must already be exported. This is what makes the developer database
# read-only for this notebook's entire run (UAT G-33-4).
import shutil
import tempfile

dev_db_path = repo_root_path / 'src' / 'fomo_db.sqlite3'
if not dev_db_path.exists():
    raise RuntimeError(f'No developer database at {dev_db_path}; run `python manage.py migrate` first.')
scratch_db_dir = Path(tempfile.mkdtemp(prefix='fomo-notebook-db-'))
scratch_db_path = scratch_db_dir / 'fomo_db.sqlite3'
shutil.copy2(dev_db_path, scratch_db_path)
os.environ['FOMO_DATABASE_PATH'] = str(scratch_db_path)

django.setup()

# This notebook intentionally imports only the campaign-coordination models/views and the
# reconciler below -- never the ephemeris view/computation modules, which trigger a large
# one-time SPICE kernel download on first import.

try:
    django.test.utils.setup_test_environment()
except RuntimeError:
    # Already set up by an earlier run of this notebook in the same kernel session.
    pass

from django.conf import settings as django_settings

resolved_db_name = django_settings.DATABASES['default']['NAME']
assert resolved_db_name == str(scratch_db_path), (
    'This notebook must never write to the developer database -- resolved DB '
    f'{resolved_db_name!r} is not the scratch copy {str(scratch_db_path)!r}.'
)

print(f'Django ready: settings module={os.environ["DJANGO_SETTINGS_MODULE"]!r}, repo_root={repo_root!r}')
print(f'Resolved database: {resolved_db_name!r} (scratch copy of the developer database)')

Django ready: settings module='src.fomo.settings', repo_root='/home/tlister/git/fomo_devel'
Resolved database: '/tmp/fomo-notebook-db-p479d9hg/fomo_db.sqlite3' (scratch copy of the developer database)


## Staff user and two test clients

Find-or-create one demo staff `User` (`is_staff=True`) with `get_or_create()` +
`set_unusable_password()`, so no real credential is ever added to the dev database. Two
`django.test.Client` instances are built from it:

- `public_client` -- never logged in. Used for the public submission form and the public
  campaign table.
- `staff_client` -- `force_login(staff_user)`. Used for the approval queue and the
  attribution queue.

Two clients, not one: `campaigns:approval_queue`, `campaigns:decide`,
`campaigns:attribution` and `campaigns:attribution_decide` are all `StaffRequiredMixin`-
gated -- using a genuinely anonymous client for the public paths (`campaigns:submit`,
`campaigns:table`) proves those paths really need no login, rather than merely asserting it.

In [2]:
from django.contrib.auth.models import User
from django.test import Client

DEMO_STAFF_USERNAME = 'campaign_lifecycle_demo_staff'

staff_user, staff_created = User.objects.get_or_create(
    username=DEMO_STAFF_USERNAME,
    defaults=dict(email='campaign-lifecycle-demo-staff@example.org', is_staff=True),
)
if staff_created:
    staff_user.set_unusable_password()
    staff_user.save()
print(
    f'Staff user: {staff_user.username!r} (pk={staff_user.pk}) '
    f'{"created" if staff_created else "found"}, is_staff={staff_user.is_staff}'
)

public_client = Client()
staff_client = Client()
staff_client.force_login(staff_user)
print('public_client: anonymous (never logged in)')
print(f'staff_client: force_login({staff_user.username!r})')

No Profile found for campaign_lifecycle_demo_staff. Creating Profile.


Staff user: 'campaign_lifecycle_demo_staff' (pk=17) created, is_staff=True
public_client: anonymous (never logged in)
staff_client: force_login('campaign_lifecycle_demo_staff')


## Demo-scoped reset

Makes the whole notebook re-runnable against any dev DB. This cell deletes ONLY this
demo's own rows: every `CampaignRun` whose `campaign` is the `Campaign Lifecycle Demo`
`TargetList`, every `CalendarEvent` whose `target_list` is that same `TargetList`, and any
`CalendarEvent` whose `url` starts with this demo's own legacy-event prefix (the
hand-entered event created further down, to demonstrate the attribution queue). It is
guarded on that `TargetList` already existing -- on a first-ever run there is nothing to
delete.

This reset now runs inside the scratch copy of the dev database made in the setup cell
above, so it protects nothing shared -- it exists purely to make the notebook re-runnable
against a fresh copy, scoped by campaign name so it never touches anything wider than the
demo campaign, and never a blanket `CampaignRun.objects.all().delete()`. This is required
because the public submission path below uses `CampaignRun.objects.create()`, which is not
idempotent and would otherwise trip one of `CampaignRun`'s two natural-key
`UniqueConstraint`s on a second execution of this notebook.

In [3]:
from tom_calendar.models import CalendarEvent
from tom_targets.models import TargetList

from solsys_code.models import CampaignRun

DEMO_CAMPAIGN_NAME = 'Campaign Lifecycle Demo'
# Namespaced distinctly from the reconciler demo's own 'RUN:' key family, and from that
# notebook's obscodes X29/X30, so the two demo notebooks never interfere with each other.
DEMO_LEGACY_EVENT_URL_PREFIX = 'LEGACY:campaign-lifecycle-demo:'

existing_demo_campaign = TargetList.objects.filter(name=DEMO_CAMPAIGN_NAME).first()
if existing_demo_campaign is None:
    print(f'No prior {DEMO_CAMPAIGN_NAME!r} campaign found -- nothing to reset (first run).')
else:
    deleted_runs, _ = CampaignRun.objects.filter(campaign=existing_demo_campaign).delete()
    deleted_campaign_events, _ = CalendarEvent.objects.filter(target_list=existing_demo_campaign).delete()
    deleted_legacy_events, _ = CalendarEvent.objects.filter(url__startswith=DEMO_LEGACY_EVENT_URL_PREFIX).delete()
    print(
        f'Reset {DEMO_CAMPAIGN_NAME!r}: deleted {deleted_runs} CampaignRun row(s) (cascade-deletes '
        f'their own RUN:-namespaced events), {deleted_campaign_events} additional CalendarEvent '
        f'row(s) still linked to the campaign, {deleted_legacy_events} legacy-prefixed CalendarEvent '
        f'row(s).'
    )

No prior 'Campaign Lifecycle Demo' campaign found -- nothing to reset (first run).


## Seed Observatory records and the campaign TargetList

Three ground `Observatory` rows, `update_or_create`d so this cell is safe to re-run, each
with a real IANA `timezone` so `sun_event()` can compute dip-corrected sunset/sunrise:

- `Y21` -- a Chilean-style classical site (`America/Santiago`).
- `Y22` -- an LCO-network-style site (`Australia/Sydney`, an LCO network node location).
- `Y23` -- a Paranal-shaped site for the ESO run, modelled on the real VLT/FORS2 site at MPC
  309 Paranal in name, coordinates and altitude, but kept under this demo's own obscode
  rather than writing to the real `309` row.

`X29`/`X30` are `reconcile_campaign_runs_demo.ipynb`'s own obscodes -- deliberately not
reused here, so the two demo notebooks never interfere with each other.

The campaign container is a `tom_targets.models.TargetList`, found-or-created by name.

In [4]:
from tom_targets.models import TargetList

from solsys_code.solsys_code_observatory.models import Observatory

y21_site, _ = Observatory.objects.update_or_create(
    obscode='Y21',
    defaults=dict(
        name='Y21 Classical Site (Campaign Lifecycle Demo)',
        short_name='Y21 Classical',
        lat=-30.1697,
        lon=-70.8065,
        altitude=2207,
        timezone='America/Santiago',
        observations_type=Observatory.OPTICAL_OBSTYPE,
    ),
)
print(f'Y21 (classical): obscode={y21_site.obscode!r}  timezone={y21_site.timezone!r}')

y22_site, _ = Observatory.objects.update_or_create(
    obscode='Y22',
    defaults=dict(
        name='Y22 LCO Network Site (Campaign Lifecycle Demo)',
        short_name='Y22 LCO Siding Spring',
        lat=-31.2733,
        lon=149.0644,
        altitude=1116,
        timezone='Australia/Sydney',
        observations_type=Observatory.OPTICAL_OBSTYPE,
    ),
)
print(f'Y22 (LCO queue): obscode={y22_site.obscode!r}  timezone={y22_site.timezone!r}')

y23_site, _ = Observatory.objects.update_or_create(
    obscode='Y23',
    defaults=dict(
        name='Y23 Paranal-shaped Site (Campaign Lifecycle Demo, modeled on real VLT/FORS2 at MPC 309)',
        short_name='Y23 VLT/FORS2-shaped',
        lat=-24.6275,
        lon=-70.4044,
        altitude=2635,
        timezone='America/Santiago',
        observations_type=Observatory.OPTICAL_OBSTYPE,
    ),
)
print(
    f'Y23 (ESO queue, resolved later via Sites Needing Review): '
    f'obscode={y23_site.obscode!r}  timezone={y23_site.timezone!r}'
)

campaign, campaign_created = TargetList.objects.get_or_create(name=DEMO_CAMPAIGN_NAME)
print(f'\nCampaign: {campaign.name!r} (pk={campaign.pk}) {"created" if campaign_created else "found"}')

Y21 (classical): obscode='Y21'  timezone='America/Santiago'
Y22 (LCO queue): obscode='Y22'  timezone='Australia/Sydney'
Y23 (ESO queue, resolved later via Sites Needing Review): obscode='Y23'  timezone='America/Santiago'

Campaign: 'Campaign Lifecycle Demo' (pk=13) created


## Four submissions through the public form

One `public_client.post()` to `campaigns:submit` per run, using the SAME anonymous client
that never logs in -- the public submission path genuinely needs no authentication. Each
POST asserts a 302 redirect to `campaigns:submission_thanks`, then looks the created run
back up by its natural key (`campaign` + `telescope_instrument`) rather than assuming a pk,
since a re-run of this notebook against a shared dev DB may not always land on the same pk
sequence.

The four submissions:

- **Classical** -- `site_raw='Y21'`, a 3-night range.
- **LCO queue** -- a distinct `telescope_instrument`, `site_raw='Y22'`, a 3-night range.
- **ESO queue** -- a VLT/FORS2-shaped `telescope_instrument` (`'UT1/FORS2'`), and a
  `site_raw` that is deliberately unresolvable free text longer than 4 characters
  (`'Paranal Observatory (site TBC)'`) so site resolution fails at approve time and the run
  lands in Sites Needing Review -- resolved later, in Task 2, via `Y23`.
- **Class-wide** -- a site-agnostic `telescope_instrument`, `site_raw` left blank, and a
  whole-month window.

Every submission supplies the required `contact_person`/`contact_email` and leaves the
`alt_contact_info` honeypot blank (by omitting it -- the form's `clean_alt_contact_info()`
treats a missing value the same as an explicit blank string). The printed output below
shows all four runs arriving as `source='web'` / `approval_status='pending_review'` with no
site and no calendar event yet, plus the growing `django.core.mail.outbox` -- the staff
notifications the view sent.

In [5]:
from django.core import mail
from django.urls import reverse

from solsys_code.models import CampaignRun

CONTACT_PERSON = 'Dr. Lifecycle Demo'
CONTACT_EMAIL = 'lifecycle-demo-contact@example.org'

SUBMISSIONS = [
    (
        'Classical',
        dict(
            campaign=campaign.pk,
            telescope_instrument='Y21 0.4m/SBIG-STX16803',
            site_raw='Y21',
            obs_date='2026-09-01 to 2026-09-03',
            contact_person=CONTACT_PERSON,
            contact_email=CONTACT_EMAIL,
        ),
    ),
    (
        'LCO queue',
        dict(
            campaign=campaign.pk,
            telescope_instrument='Y22 1m0-SciCam-Sinistro',
            site_raw='Y22',
            obs_date='2026-09-05 to 2026-09-07',
            contact_person=CONTACT_PERSON,
            contact_email=CONTACT_EMAIL,
        ),
    ),
    (
        'ESO queue',
        dict(
            campaign=campaign.pk,
            telescope_instrument='UT1/FORS2',
            site_raw='Paranal Observatory (site TBC)',
            obs_date='2026-09-10 to 2026-09-12',
            contact_person=CONTACT_PERSON,
            contact_email=CONTACT_EMAIL,
        ),
    ),
    (
        'Class-wide',
        dict(
            campaign=campaign.pk,
            telescope_instrument='LCO 1m0 Network (Campaign Lifecycle Demo)',
            site_raw='',
            obs_date='2026-09-01 to 2026-09-30',
            contact_person=CONTACT_PERSON,
            contact_email=CONTACT_EMAIL,
        ),
    ),
]

submitted_runs = {}
for label, form_data in SUBMISSIONS:
    response = public_client.post(reverse('campaigns:submit'), data=form_data)
    assert (
        response.status_code == 302
    ), f'{label} submission failed: {response.status_code} {getattr(response, "context", None)}'
    assert response.url == reverse('campaigns:submission_thanks')
    run = CampaignRun.objects.get(campaign=campaign, telescope_instrument=form_data['telescope_instrument'])
    submitted_runs[label] = run
    print(
        f'{label:<12} pk={run.pk}  source={run.source!r}  approval_status={run.approval_status!r}  '
        f'site_raw={run.site_raw!r}  window={run.window_start}..{run.window_end}  '
        f'mail.outbox so far: {len(mail.outbox)}'
    )

classical_run = submitted_runs['Classical']
lco_queue_run = submitted_runs['LCO queue']
eso_queue_run = submitted_runs['ESO queue']
class_wide_run = submitted_runs['Class-wide']

Note: NumExpr detected 32 cores but "NUMEXPR_MAX_THREADS" not set, so enforcing safe limit of 16.


NumExpr defaulting to 16 threads.


Using fallback library next to module: /home/tlister/venv/devel_fomo311_venv/lib/python3.11/site-packages/spiceypy/utils/libcspice.so


registering new views: args: ('groups', <class 'tom_common.api_views.GroupViewSet'>, 'groups'), kwargs: {}


registering new views: args: ('targets', <class 'tom_targets.api_views.TargetViewSet'>, 'targets'), kwargs: {}


...

registering new views: args: ('targetlist', <class 'tom_targets.api_views.TargetListViewSet'>, 'targetlist'), kwargs: {}


registering new views: args: ('observations', <class 'tom_observations.api_views.ObservationRecordViewSet'>, 'observations'), kwargs: {}


registering new views: args: ('dataproducts', <class 'tom_dataproducts.api_views.DataProductViewSet'>, 'dataproducts'), kwargs: {}


registering new views: args: ('reduceddatums', <class 'tom_dataproducts.api_views.ReducedDatumViewSet'>, 'reduceddatums'), kwargs: {}


Classical    pk=69  source='web'  approval_status='pending_review'  site_raw='Y21'  window=2026-09-01..2026-09-03  mail.outbox so far: 1
LCO queue    pk=70  source='web'  approval_status='pending_review'  site_raw='Y22'  window=2026-09-05..2026-09-07  mail.outbox so far: 2
ESO queue    pk=71  source='web'  approval_status='pending_review'  site_raw='Paranal Observatory (site TBC)'  window=2026-09-10..2026-09-12  mail.outbox so far: 3
Class-wide   pk=72  source='web'  approval_status='pending_review'  site_raw=''  window=2026-09-01..2026-09-30  mail.outbox so far: 4


## Provenance stamping, with the honesty note

The public submission form exposes neither `source` nor `telescope_class` -- in production
those values come from the ingest path that CREATED the row. `import_campaign_csv` already
writes `source=CSV_IMPORT`; v2.3's ADAPT-01..03 will rewire the three calendar-sync
adapters to write `LCO_QUEUE`/`GEMINI_QUEUE`/`ESO_QUEUE`. Those adapter paths do not exist
yet, so this notebook stamps the four values by hand -- purely to obtain four rows of the
right provenance for the walkthrough below. **This is not a shortcut for the workflow
itself**: every *state transition* below (submission, approval, site resolution,
attribution) still goes through a real staff-facing view (D-01); only these two fields,
which the public form was never given a way to set, are written directly here.

`telescope_class` must be set on the class-wide run BEFORE it is approved in Task 2 below,
because `CampaignRunDecisionView.post()`'s approve branch computes `site_needs_review =
needs_review and not run.telescope_class`, and `reconcile_run()` dispatches on
`telescope_class` first.

In [6]:
classical_run.source = CampaignRun.Source.CLASSICAL_FILE
classical_run.save(update_fields=['source'])

lco_queue_run.source = CampaignRun.Source.LCO_QUEUE
lco_queue_run.save(update_fields=['source'])

eso_queue_run.source = CampaignRun.Source.ESO_QUEUE
eso_queue_run.save(update_fields=['source'])

class_wide_run.source = CampaignRun.Source.LEGACY
class_wide_run.telescope_class = CampaignRun.TelescopeClass.ONE_M0
class_wide_run.save(update_fields=['source', 'telescope_class'])

for label, run in submitted_runs.items():
    run.refresh_from_db()
    print(f'{label:<12} pk={run.pk}  source={run.source!r}  telescope_class={run.telescope_class!r}')

Classical    pk=69  source='classical_file'  telescope_class=''
LCO queue    pk=70  source='lco_queue'  telescope_class=''
ESO queue    pk=71  source='eso_queue'  telescope_class=''
Class-wide   pk=72  source='legacy'  telescope_class='1m0'


## The approval queue, before

`staff_client.get(reverse('campaigns:approval_queue'))` -- gated by `StaffRequiredMixin`,
so only a logged-in staff user can reach it. The view builds its site-candidate pool
(`build_site_candidates()`) once per request; that pool is 24h-cached and falls back to a
local-only pool on any MPC API failure, so a cold cache may make one outbound MPC call here
but never fails. Sites Needing Review renders first on this page (plan 27-07) -- empty for
now, since none of the four demo runs has been approved yet.

In [7]:
response = staff_client.get(reverse('campaigns:approval_queue'))
assert response.status_code == 200

pending_pks = {run.pk for run in response.context['pending_table'].data}
review_pks = {run.pk for run in response.context['review_table'].data}
print(f'Pending review (pending_table): {sorted(pending_pks)}')
print(f'Needing site review (review_table), before any approval: {sorted(review_pks)}')

demo_pks = {run.pk for run in submitted_runs.values()}
assert demo_pks <= pending_pks, 'All four demo runs should still be pending review'

Pending review (pending_table): [69, 70, 71, 72]
Needing site review (review_table), before any approval: [45]


## Approve all four

One `staff_client.post(reverse('campaigns:decide', args=[run.pk]), {'action': 'approve'})`
per run -- no `site_selection` in the POST body, so `_resolve_site` falls back to each
run's own `site_raw`. The executed output below makes three facts visible:

- The `Y21`/`Y22` runs resolve their sites via tier 1 (an exact local `Observatory.obscode`
  match, no network) and immediately get their per-night events, because `approve()` calls
  `reconcile_run()` itself.
- The ESO run is approved anyway, with `site=None` and `site_needs_review=True` -- site
  resolution failure never blocks approval -- and therefore has no events yet.
- The class-wide run gets its container event with `site` still `None` and
  `site_needs_review` still `False`, because its `telescope_class` (set in Task 1) already
  answers "why is there no site" (models.py D-06) -- it was never eligible for the Sites
  Needing Review queue in the first place.

In [8]:
from solsys_code.allocation_projector import allocation_events
from solsys_code.campaign_reconciler import owned_events


def _event_count(run):
    # Phase 37 Plan 07 fix: a run's calendar events live in ONE of two namespaces --
    # owned_events() (the whole-window `RUN:` container) or allocation_events() (the
    # per-night `ALLOC:` family) -- never both, so summing both queryset counts is
    # always exactly the count owned_events() alone used to (wrongly) claim was zero
    # for a run that actually dispatches to the ALLOC: family. See 'The four-way
    # payoff' below for which run takes which branch.
    return owned_events(run).count() + allocation_events(run).count()


for label, run in submitted_runs.items():
    response = staff_client.post(reverse('campaigns:decide', args=[run.pk]), {'action': 'approve'})
    assert response.status_code == 302
    assert response.url == reverse('campaigns:approval_queue')
    run.refresh_from_db()
    print(
        f'{label:<12} pk={run.pk}  approval_status={run.approval_status!r}  site={run.site!r}  '
        f'site_needs_review={run.site_needs_review}  events={_event_count(run)}'
    )

assert classical_run.site is not None and not classical_run.site_needs_review
assert lco_queue_run.site is not None and not lco_queue_run.site_needs_review
assert eso_queue_run.site is None and eso_queue_run.site_needs_review
assert class_wide_run.site is None and not class_wide_run.site_needs_review

Classical    pk=69  approval_status='approved'  site=<Observatory: Y21: Y21 Classical Site (Campaign Lifecycle Demo)>  site_needs_review=False  events=3
LCO queue    pk=70  approval_status='approved'  site=<Observatory: Y22: Y22 LCO Network Site (Campaign Lifecycle Demo)>  site_needs_review=False  events=1
ESO queue    pk=71  approval_status='approved'  site=None  site_needs_review=True  events=0
Class-wide   pk=72  approval_status='approved'  site=None  site_needs_review=False  events=1


## Site-review resolution

GET the approval queue again: the ESO run now appears in Sites Needing Review. Then
`staff_client.post(reverse('campaigns:decide', args=[eso_queue_run.pk]), {'action':
'resolve_site', 'site_selection': 'Y23'})` resolves it against the Paranal-shaped `Y23`
site seeded in Task 1. `_resolve_site()` clears `site_needs_review` only AFTER
`reconcile_run()` succeeds -- so a failed projection would leave the row in the queue for a
retry rather than silently losing it.

In [9]:
response = staff_client.get(reverse('campaigns:approval_queue'))
assert response.status_code == 200
review_pks_after_approve = {run.pk for run in response.context['review_table'].data}
print(f'Needing site review, after approval: {sorted(review_pks_after_approve)}')
assert eso_queue_run.pk in review_pks_after_approve

response = staff_client.post(
    reverse('campaigns:decide', args=[eso_queue_run.pk]),
    {'action': 'resolve_site', 'site_selection': 'Y23'},
)
assert response.status_code == 302
eso_queue_run.refresh_from_db()
print(
    f'ESO queue run pk={eso_queue_run.pk}  site={eso_queue_run.site!r}  '
    f'site_needs_review={eso_queue_run.site_needs_review}  events={owned_events(eso_queue_run).count()}'
)
assert eso_queue_run.site == y23_site
assert not eso_queue_run.site_needs_review
assert owned_events(eso_queue_run).count() > 0

Needing site review, after approval: [45, 71]
ESO queue run pk=71  site=<Observatory: Y23: Y23 Paranal-shaped Site (Campaign Lifecycle Demo, modeled on real VLT/FORS2 at MPC 309)>  site_needs_review=False  events=1


## An orphan calendar event, and the attribution queue

A `load_telescope_runs`-style entry that predates the canonical run record is represented
here by creating one `CalendarEvent` directly -- a hand-entered/legacy-sync-created row
with NO `CalendarEventMeta` companion row at all, which is precisely the orphan kind
`orphan_calendar_events()`'s first branch exists to catch. Its `target_list` is set to the
demo campaign (required -- `_eligible_runs_for_event()` hard-gates on it), its
`telescope`/`instrument` match the classical run's own two halves, and its window overlaps
the classical run's window. Its `url` uses this demo's own legacy prefix, which the Task 1
reset cell already cleans up.

`staff_client.get(reverse('campaigns:attribution'))` confirms it is now in the backlog.
Its scored candidate list is then printed by calling `campaign_attribution.
candidates_for_event()` directly -- the same function the view's context uses -- rather
than depending on which pagination page it happens to land on among any other pre-existing
orphans already in this dev DB. Three scoring signals: date overlap (weight 0.40),
instrument similarity (0.35) and telescope match (0.25). Bands are High (>=0.75), Medium
(>=0.50), else Low. The hard campaign boundary: only runs in the SAME campaign as the event
are ever scored at all (`_eligible_runs_for_event()`).

In [10]:
from datetime import datetime
from datetime import timezone as dt_timezone

from tom_calendar.models import CalendarEvent

from solsys_code import campaign_attribution

classical_telescope, _sep, classical_instrument = classical_run.telescope_instrument.partition('/')

orphan_event = CalendarEvent.objects.create(
    title=f'{campaign.name}: hand-entered pre-canon night (demo)',
    description='A pre-existing, hand-entered calendar entry that predates the canonical CampaignRun record.',
    start_time=datetime(2026, 9, 1, 22, 0, tzinfo=dt_timezone.utc),
    end_time=datetime(2026, 9, 2, 10, 0, tzinfo=dt_timezone.utc),
    url=f'{DEMO_LEGACY_EVENT_URL_PREFIX}2026-09-01',
    target_list=campaign,
    telescope=classical_telescope,
    instrument=classical_instrument,
)
print(
    f'Orphan event created: pk={orphan_event.pk}  url={orphan_event.url!r}  '
    f'telescope={orphan_event.telescope!r}  instrument={orphan_event.instrument!r}'
)
assert not hasattr(orphan_event, 'telescope_label_meta'), 'orphan event must have no CalendarEventMeta yet'

response = staff_client.get(reverse('campaigns:attribution'))
assert response.status_code == 200
print(f'attribution_count={response.context["attribution_count"]}  is_drained={response.context["is_drained"]}')

candidates = campaign_attribution.candidates_for_event(orphan_event)
print(f'Orphan event pk={orphan_event.pk} candidates:')
for candidate in candidates:
    print(
        f'  run pk={candidate.run.pk}  source={candidate.run.source!r}  '
        f'score={candidate.score:.2f}  band={candidate.band}'
    )
assert any(c.run.pk == classical_run.pk for c in candidates), 'classical_run should be a scored candidate'

Orphan event created: pk=364  url='LEGACY:campaign-lifecycle-demo:2026-09-01'  telescope='Y21 0.4m'  instrument='SBIG-STX16803'


attribution_count=30  is_drained=False
Orphan event pk=364 candidates:
  run pk=69  source='classical_file'  score=0.82  band=high
  run pk=72  source='legacy'  score=0.58  band=medium
  run pk=70  source='lco_queue'  score=0.21  band=low
  run pk=71  source='eso_queue'  score=0.20  band=low


## D-07: the renamed FTN/FTS/SOAR telescope labels still resolve to a site-level attribution match

Plan 34-02 renamed the LCO 2m0/4m0 `SITE_TELESCOPE_MAP` labels from `COJ-2m0`/`OGG-2m0`/
`SOR-4m0` to `FTN`/`FTS`/`SOAR`. `campaign_attribution.telescope_match_score()` resolves
these three renamed labels straight to an obscode via its LABEL-keyed
`OBSERVED_TELESCOPE_OBSCODES` table (`{'FTN': 'F65', 'FTS': 'E10', 'SOAR': 'I33'}`) --
`_extract_lco_site_code()`'s site-code bridge (splitting on `-`) can never recover a
site code from these three labels at all, and is not consulted by this path. This cell
exists because the paired-docs rule fires on that renaming (CLAUDE.md), and belongs in
this notebook rather than in 34-02 because its own tasks run entirely against the
in-memory Django test database, not this notebook's scratch copy of the developer
database.

**Fixed 2026-09-19 (Phase 37 Plan 07):** an earlier version of this cell asserted
`_extract_lco_site_code('FTN') == 'ogg'` via the now-unused
`calendar_utils.OBSERVED_TELESCOPE_SITE_CODES` table -- that assertion could never pass
against the current code, because `_extract_lco_site_code()` returns `None` for all
three renamed labels (they carry no leading three-letter site-code token a plain split
can find) and the label-keyed `OBSERVED_TELESCOPE_OBSCODES` table is what actually
resolves them now. The cell below demonstrates the real, current bridge directly.

In [11]:
from datetime import date as date_cls

from solsys_code.campaign_attribution import OBSERVED_TELESCOPE_OBSCODES, TELESCOPE_MATCH_SITE, telescope_match_score
from solsys_code.models import CampaignRun
from solsys_code.solsys_code_observatory.models import Observatory
from tom_targets.models import TargetList

for telescope_label, expected_obscode in OBSERVED_TELESCOPE_OBSCODES.items():
    print(f'{telescope_label:<5} -> obscode {expected_obscode!r}')
    assert OBSERVED_TELESCOPE_OBSCODES[telescope_label] == expected_obscode

# A dedicated demo-only campaign (never `campaign`, the one the rest of this notebook's
# public-table/attribution cells assert an exact membership set against) keeps this
# demo run from showing up in those unrelated assertions.
site_match_demo_campaign, _ = TargetList.objects.get_or_create(name='D-07 site-match demo campaign')

# One concrete site-level match, all the way through telescope_match_score(): a
# demo-only run at ogg's real F65 Observatory (Haleakala), scored against an orphan
# event whose telescope is the renamed 'FTN' label.
ftn_site, _ = Observatory.objects.get_or_create(
    obscode='F65',
    defaults=dict(
        name='Faulkes Telescope North',
        short_name='FTN',
        lat=20.7075,
        lon=-156.258,
        altitude=3037,
        timezone='Pacific/Honolulu',
    ),
)
ftn_demo_run, _ = CampaignRun.objects.update_or_create(
    campaign=site_match_demo_campaign,
    telescope_instrument='FTN site-match demo (D-07 attribution)',
    window_start=date_cls(2026, 9, 1),
    window_end=date_cls(2026, 9, 1),
    defaults=dict(
        site=ftn_site,
        site_raw='F65',
        approval_status=CampaignRun.ApprovalStatus.APPROVED,
        source=CampaignRun.Source.LCO_QUEUE,
    ),
)
match_score, evidence = telescope_match_score(ftn_demo_run, telescope_code='FTN', instrument_code='2M0-SCICAM-MUSCAT')
print()
print(f'FTN match level: {match_score} ({evidence})')
assert match_score == TELESCOPE_MATCH_SITE, f'expected a site-level match, got {match_score}'

FTN   -> obscode 'F65'
FTS   -> obscode 'E10'
SOAR  -> obscode 'I33'

FTN match level: 1.0 (orphan observed telescope 'FTN' resolves to obscode F65, matching the run's site obscode F65)


## A fifth, deliberately-rejected submission (attribution exclusion demo)

A fifth `CampaignRun` exists here ONLY to demonstrate that a rejected run is never offered
as an attribution match (27-REVIEW IN-02, D-01/D-02/D-03) -- it is deliberately NOT part of
the four-run lifecycle narrative the rest of this notebook follows, and is never added to
`submitted_runs`, so the approve-all cell above, the four-way payoff cell below and the
Summary cell all stay untouched by it. It is submitted through the same public form
(`campaigns:submit`) as the other four, so it arrives as `pending_review` just like them,
with a `telescope_instrument`/`site_raw` matching the orphan event so it would otherwise
score highly, and a window that overlaps the orphan event but differs from `classical_run`'s
own window so the two rows don't collide on the campaign+telescope_instrument+window
natural key.

The cell below shows it as a scored candidate for `orphan_event` first, while it is still
`pending_review` (the non-vacuous control -- proving the fixture really can produce a
candidate, so the exclusion below is about the rejection, not about the fixture), then
rejects it through the same staff decision endpoint (`campaigns:decide`,
`action=reject`) the four-way approvals above use, and shows it absent from the candidate
list second, with `classical_run` still present -- so the exclusion is visibly targeted at
`approval_status` alone, not a blanket emptying of the candidate list.


In [12]:
from datetime import date as date_cls

response = public_client.post(
    reverse('campaigns:submit'),
    data=dict(
        campaign=campaign.pk,
        telescope_instrument='Y21 0.4m/SBIG-STX16803',
        site_raw='Y21',
        obs_date='2026-09-01 to 2026-09-02',
        contact_person=CONTACT_PERSON,
        contact_email=CONTACT_EMAIL,
    ),
)
assert response.status_code == 302
assert response.url == reverse('campaigns:submission_thanks')

# Disambiguated by window, not just campaign+telescope_instrument: classical_run already
# carries the same telescope_instrument text with a different (2026-09-01..2026-09-03)
# window, so a lookup without window_start/window_end would raise MultipleObjectsReturned.
rejected_demo_run = CampaignRun.objects.get(
    campaign=campaign,
    telescope_instrument='Y21 0.4m/SBIG-STX16803',
    window_start=date_cls(2026, 9, 1),
    window_end=date_cls(2026, 9, 2),
)
print(
    f'Rejected demo run (5th -- exists only for the attribution-exclusion demo): '
    f'pk={rejected_demo_run.pk}  source={rejected_demo_run.source!r}  '
    f'approval_status={rejected_demo_run.approval_status!r}'
)

candidates_before_reject = campaign_attribution.candidates_for_event(orphan_event)
candidate_pks_before_reject = {c.run.pk for c in candidates_before_reject}
print(
    f'Rejected demo run (5th) -- candidate pks for orphan_event while pending_review: '
    f'{sorted(candidate_pks_before_reject)}'
)
assert rejected_demo_run.pk in candidate_pks_before_reject, (
    'Rejected demo run (5th) must be a scored candidate BEFORE rejection -- the '
    'non-vacuous control proving the exclusion below is about approval_status, not the '
    'fixture.'
)

response = staff_client.post(
    reverse('campaigns:decide', args=[rejected_demo_run.pk]),
    {'action': 'reject'},
)
assert response.status_code == 302
rejected_demo_run.refresh_from_db()
assert rejected_demo_run.approval_status == CampaignRun.ApprovalStatus.REJECTED
print(f'Rejected demo run (5th) -- rejected: ' f'approval_status={rejected_demo_run.approval_status!r}')

candidates_after_reject = campaign_attribution.candidates_for_event(orphan_event)
candidate_pks_after_reject = {c.run.pk for c in candidates_after_reject}
print(
    f'Rejected demo run (5th) -- candidate pks for orphan_event after rejection: '
    f'{sorted(candidate_pks_after_reject)}'
)
assert rejected_demo_run.pk not in candidate_pks_after_reject, (
    'Rejected demo run (5th) must be ABSENT from candidates after rejection ' '(27-REVIEW IN-02).'
)
assert classical_run.pk in candidate_pks_after_reject, (
    'classical_run must still be a candidate -- the exclusion above is targeted, not a '
    'blanket emptying of the candidate list.'
)

Rejected demo run (5th -- exists only for the attribution-exclusion demo): pk=74  source='web'  approval_status='pending_review'
Rejected demo run (5th) -- candidate pks for orphan_event while pending_review: [69, 70, 71, 72, 74]
Rejected demo run (5th) -- rejected: approval_status='rejected'
Rejected demo run (5th) -- candidate pks for orphan_event after rejection: [69, 70, 71, 72]


## Confirm the attribution

`staff_client.post(reverse('campaigns:attribution_decide'), {'action': 'confirm', 'kind':
'event', 'orphan_pk': orphan_event.pk, 'run_pk': classical_run.pk})` writes the
confirmation. The view re-derives eligibility server-side via `campaign_attribution.
is_offered_candidate()` before writing -- a tampered POST cannot create an association
across a campaign boundary. This attribution also survives future reconciler sweeps:
`_detach_stale_family_events()` is scoped to the classical run's own `RUN:{pk}` url
namespace, and this orphan event's url is outside it entirely.

In [13]:
from solsys_code.models import CalendarEventMeta

response = staff_client.post(
    reverse('campaigns:attribution_decide'),
    {'action': 'confirm', 'kind': 'event', 'orphan_pk': orphan_event.pk, 'run_pk': classical_run.pk},
)
assert response.status_code == 302

meta = CalendarEventMeta.objects.get(event_id=orphan_event.pk)
print(
    f'CalendarEventMeta for event pk={orphan_event.pk}: run={meta.run!r}  '
    f'confirmed_by={meta.confirmed_by!r}  confirmed_at={meta.confirmed_at!r}'
)
assert meta.run_id == classical_run.pk
assert meta.confirmed_by_id == staff_user.pk
assert meta.confirmed_at is not None

response = staff_client.get(reverse('campaigns:attribution'))
assert response.status_code == 200
print(f'attribution_count after confirming: {response.context["attribution_count"]}')

CalendarEventMeta for event pk=364: run=<CampaignRun: #69 Campaign Lifecycle Demo | Y21 0.4m/SBIG-STX16803 | 2026-09-01..2026-09-03 | Y21>  confirmed_by=<User: campaign_lifecycle_demo_staff>  confirmed_at=datetime.datetime(2026, 9, 21, 17, 2, 31, 864482, tzinfo=datetime.timezone.utc)


attribution_count after confirming: 29


## Campaign decoration, rendered from the link

GET the confirmed orphan event's modal (`calendar:update-event`) through `public_client` --
the same anonymous client used throughout this notebook. `campaign_decoration()` renders the
attribution from `CalendarEventMeta.run` at request time: the printed HTML fragment below shows
the "Attributed campaign run" label, the campaign name, the telescope/instrument, the window and
the run status, plus the campaign-table link anchored to the run's own row (`#run-{pk}`).


In [14]:
response = public_client.get(reverse('calendar:update-event', args=[orphan_event.pk]))
assert response.status_code == 200
content = response.content.decode()

assert 'Attributed campaign run' in content
assert campaign.name in content
assert classical_run.telescope_instrument in content
run_anchor = f'#run-{classical_run.pk}'
assert run_anchor in content

print('Decoration rendered on the event modal (GET calendar:update-event):')
print(f'  "Attributed campaign run" label present: {"Attributed campaign run" in content}')
print(f'  Campaign name ({campaign.name!r}) present: {campaign.name in content}')
print(
    f'  Telescope/instrument ({classical_run.telescope_instrument!r}) present: '
    f'{classical_run.telescope_instrument in content}'
)
print(f'  Campaign-table link anchored to {run_anchor!r}: {run_anchor in content}')

Decoration rendered on the event modal (GET calendar:update-event):
  "Attributed campaign run" label present: True
  Campaign name ('Campaign Lifecycle Demo') present: True
  Telescope/instrument ('Y21 0.4m/SBIG-STX16803') present: True
  Campaign-table link anchored to '#run-69': True


## The run tally in the calendar pop-up (Phase 37 TALLY-01)

The same "Attributed campaign run" block now also renders a live tally -- linked
groups/records plus the four ordered night segments (`[O]`/`[S]`/`[X/F]`/`[U]`) -- via
`calendar_display_extras.run_tally()`, reading `campaign_tally.get_or_compute_tally()`.
GET the same orphan event's modal again with `public_client` and show the tally line
inside the block. `classical_run` has no linked observation groups/records, but its own
already-elapsed `ALLOC:` window (2026-09-01..2026-09-03, seeded at submission time above)
means its unused-night count is EXACT, not an estimate -- the same rule "An unused
awarded night" further down decorates on the calendar itself.

In [15]:
response = public_client.get(reverse('calendar:update-event', args=[orphan_event.pk]))
assert response.status_code == 200
content = response.content.decode()

assert 'Attributed campaign run' in content
for marker in ('[O]', '[S]', '[X/F]', '[U]'):
    assert marker in content, f'{marker!r} missing from the pop-up tally line'

from solsys_code import campaign_tally

tally = campaign_tally.get_or_compute_tally(classical_run)
print(f"classical_run tally: {tally['groups']} group(s), {tally['records']} record(s)")
for segment in campaign_tally.tally_segments(tally):
    print(
        f"  {segment['marker']} {segment['label']}: {segment['count']} "
        f"(estimate={segment['is_estimate']}, known={segment['known']})"
    )

assert tally['nights_unused'] == 3, 'the three already-elapsed ALLOC: nights are all unused'
assert (
    tally['unused_is_estimate'] is False
), 'an allocation run with its own ALLOC: events gets an EXACT unused count, never an estimate'
print()
print(
    "Pop-up tally line present on the attributed event modal, agreeing with the table's "
    'Progress column by construction (D-15).'
)

classical_run tally: 0 group(s), 0 record(s)
  [O] Observed: 0 (estimate=False, known=True)
  [S] Scheduled: 0 (estimate=False, known=True)
  [X/F] Expired/failed: 0 (estimate=False, known=True)
  [U] Unused awarded night: 3 (estimate=False, known=True)

Pop-up tally line present on the attributed event modal, agreeing with the table's Progress column by construction (D-15).


## Criterion 3: the decoration survives a from-scratch rewrite

Assign a brand-new title and description to the orphan event and save it directly -- standing
in for a base-layer re-projection overwriting an event's own fields. GET the modal and the month
calendar again: the decoration is still rendered, because it comes from the link, not from
anything written into the event, so a from-scratch rewrite of the event's own fields cannot
erase it.


In [16]:
new_title = 'Rewritten from scratch (Campaign Lifecycle Demo criterion 3)'
new_description = "This text replaces the event's original description entirely."
orphan_event.title = new_title
orphan_event.description = new_description
orphan_event.save(update_fields=['title', 'description'])

response = public_client.get(reverse('calendar:update-event', args=[orphan_event.pk]))
assert response.status_code == 200
content = response.content.decode()
assert 'Attributed campaign run' in content
assert campaign.name in content

month_response = public_client.get(
    reverse('calendar:calendar'),
    {'year': orphan_event.start_time.year, 'month': orphan_event.start_time.month},
)
assert month_response.status_code == 200
month_content = month_response.content.decode()
assert 'cal-campaign-chip' in month_content

print(f'Event title after rewrite: {orphan_event.title!r}')
print(f'Event description after rewrite: {orphan_event.description!r}')
print(f'Modal still shows "Attributed campaign run": {"Attributed campaign run" in content}')
print(f'Month view still shows the campaign chip marker: {"cal-campaign-chip" in month_content}')

Event title after rewrite: 'Rewritten from scratch (Campaign Lifecycle Demo criterion 3)'
Event description after rewrite: "This text replaces the event's original description entirely."
Modal still shows "Attributed campaign run": True
Month view still shows the campaign chip marker: True


## Criterion 4: unlinking removes only the decoration

`unlink_event_from_run()` is the single writer that clears an event's attribution -- `run`,
`confirmed_by` and `confirmed_at` together -- without touching the `CalendarEvent` itself or
deleting anything. Calling it for the orphan event / classical run pair, then re-fetching both
rows and the modal, shows exactly that: the `CalendarEventMeta` row still exists with its link
cleared and `is_verified` unchanged, the `CalendarEvent` still exists with its (already-rewritten)
fields unchanged, and a fresh modal GET renders no decoration.


In [17]:
from solsys_code.campaign_utils import unlink_event_from_run

events_before_unlink = CalendarEvent.objects.count()
changed_count = unlink_event_from_run(orphan_event, classical_run)
print(f'unlink_event_from_run() cleared {changed_count} companion row(s)')

meta.refresh_from_db()
print(
    f'CalendarEventMeta after unlink: run={meta.run!r}  confirmed_by={meta.confirmed_by!r}  '
    f'confirmed_at={meta.confirmed_at!r}  is_verified={meta.is_verified}'
)
assert meta.run is None
assert meta.confirmed_by is None
assert meta.confirmed_at is None
assert meta.is_verified is True

orphan_event.refresh_from_db()
print(f'CalendarEvent after unlink: title={orphan_event.title!r}  description={orphan_event.description!r}')
assert orphan_event.title == new_title
assert orphan_event.description == new_description

events_after_unlink = CalendarEvent.objects.count()
print(f'CalendarEvent.objects.count() before unlink: {events_before_unlink}')
print(f'CalendarEvent.objects.count() after unlink:  {events_after_unlink}')
assert events_after_unlink == events_before_unlink, 'unlink must never delete a CalendarEvent'

response = public_client.get(reverse('calendar:update-event', args=[orphan_event.pk]))
assert response.status_code == 200
content = response.content.decode()
print(f'Modal shows "Attributed campaign run" after unlink: {"Attributed campaign run" in content}')
assert 'Attributed campaign run' not in content

unlink_event_from_run() cleared 1 companion row(s)
CalendarEventMeta after unlink: run=None  confirmed_by=None  confirmed_at=None  is_verified=True
CalendarEvent after unlink: title='Rewritten from scratch (Campaign Lifecycle Demo criterion 3)'  description="This text replaces the event's original description entirely."
CalendarEvent.objects.count() before unlink: 241
CalendarEvent.objects.count() after unlink:  241
Modal shows "Attributed campaign run" after unlink: False


## Series-identity link fields: observation_record and observation_group

`CalendarEventMeta` also carries `observation_record` (one-to-one) and `observation_group`
(foreign key) -- the carrier fields Phase 34's observation projector will write. No code writes
them yet in this phase; Phase 34 is the one that will. This cell links the now-unattributed
orphan event's `CalendarEventMeta` row to a demo `ObservationRecord` by hand, purely to show the
row is still offered by the event attribution queue exactly as any other unattributed event
(D-15) -- the queue keys off the attribution link (`run`) only, never the observation link.


In [18]:
from tom_observations.models import ObservationGroup, ObservationRecord
from tom_targets.models import Target
from tom_targets.tests.factories import NonSiderealTargetFactory

from solsys_code import campaign_attribution

DEMO_OBSERVATION_TARGET_NAME = 'Campaign Lifecycle Demo Series-Identity Target'
DEMO_OBSERVATION_GROUP_NAME = 'Campaign Lifecycle Demo Group'

observation_target = Target.objects.filter(name=DEMO_OBSERVATION_TARGET_NAME).first()
if observation_target is None:
    observation_target = NonSiderealTargetFactory.create(name=DEMO_OBSERVATION_TARGET_NAME)

observation_record, _ = ObservationRecord.objects.get_or_create(
    facility='LCO',
    observation_id='campaign-lifecycle-demo-obs-1',
    defaults=dict(
        target=observation_target,
        status='COMPLETED',
        parameters={'instrument_type': '2M0-SCICAM-MUSCAT'},
    ),
)
observation_group, _ = ObservationGroup.objects.get_or_create(name=DEMO_OBSERVATION_GROUP_NAME)
observation_group.observation_records.add(observation_record)

meta.observation_record = observation_record
meta.observation_group = observation_group
meta.save(update_fields=['observation_record', 'observation_group'])
meta.refresh_from_db()
print(
    f'CalendarEventMeta pk={meta.pk}: observation_record={meta.observation_record!r}  '
    f'observation_group={meta.observation_group!r}  run={meta.run!r}'
)
assert meta.run is None, 'this cell demonstrates the observation link, not attribution -- run stays unset'

orphan_pks = {event.pk for event in campaign_attribution.orphan_calendar_events()}
print(f'orphan_event.pk={orphan_event.pk} in orphan_calendar_events(): {orphan_event.pk in orphan_pks}')
assert orphan_event.pk in orphan_pks, (
    'an event with observation_record set but run unset must still be offered by the '
    'attribution queue, exactly like any other unattributed event (D-15)'
)

Target post save hook: Campaign Lifecycle Demo Series-Identity Target created: True


Target post save hook: Campaign Lifecycle Demo Series-Identity Target created: False


unprojectable observation_id='campaign-lifecycle-demo-obs-1': KeyError: 'start'


Observation change state hook: Campaign Lifecycle Demo Series-Identity Target @ LCO from None to COMPLETED


unprojectable observation_id='campaign-lifecycle-demo-obs-1': KeyError: 'start'


group membership change left observation_id='campaign-lifecycle-demo-obs-1' unprojectable: KeyError


CalendarEventMeta pk=364: observation_record=<ObservationRecord: Campaign Lifecycle Demo Series-Identity Target @ LCO>  observation_group=<ObservationGroup: Campaign Lifecycle Demo Group>  run=None
orphan_event.pk=364 in orphan_calendar_events(): True


## The four-way payoff

This is the cell the whole notebook exists for. Looping over all four runs, in submission
order, and printing every calendar event each run actually owns -- `owned_events(run)` (the
`RUN:` namespace, the whole-window container) plus `allocation_events(run)` (the `ALLOC:`
namespace, the per-night family) -- ordered by `start_time`. Reading both namespaces is
required (Phase 37 Plan 07 fix, see below): a run owns events in exactly ONE of the two, so
summing both queryset counts is always the true total.

`campaign_reconciler.dispatches_per_night()` is the single decision point for which family a
run belongs to: a class-wide run (`telescope_class` set), a satellite-sited run, or a
queue-scheduled run (`source` in `{lco_queue, soar_queue, gemini_queue, eso_queue}`) all take
the whole-window `RUN:{pk}` container -- one event spanning the whole window, refreshed in
place on every reconcile, never split into per-night rows. Everything else -- a non-queue,
non-satellite, non-class-wide run with a resolved ground site, which is exactly what a
classical schedule import looks like -- takes the per-night `ALLOC:{pk}:{night}` family: one
sunset-to-sunrise event per observing night, with a real dip-corrected dark window computed
once at creation. So three of these four runs below (the LCO-queue run, the ESO-queue run and
the class-wide run) render as a single whole-window container despite two of them carrying a
real ground site and a real window -- only the classical run gets the richer per-night
treatment `load_telescope_runs_demo.ipynb` and `reconcile_campaign_runs_demo.ipynb` cover in
depth. See "How do I get every campaign run onto the calendar?" in
`docs/runbooks/telescope_runs_calendar.rst` for the operator-facing version of this rule.

**Fixed 2026-09-19 (Phase 37 Plan 07):** this cell's earlier committed output was stale --
`_extract_lco_site_code()`/`owned_events()`-only counting predates a later change to
`dispatches_per_night()`'s dispatch rule (this classical run now dispatches to the `ALLOC:`
family, not the old per-night `RUN:` family), so the cell's own assertions no longer matched
real behaviour when re-executed. The cell below now reads both namespaces and asserts the
current, real dispatch outcome directly.

In [19]:
import re

from solsys_code.allocation_projector import allocation_events

ALLOC_NIGHT_RE = re.compile(r'^ALLOC:(\d+):\d{4}-\d{2}-\d{2}$')

per_night_run_pks = set()
container_only_run_pks = set()

for label, run in [
    ('Classical', classical_run),
    ('LCO queue', lco_queue_run),
    ('ESO queue', eso_queue_run),
    ('Class-wide', class_wide_run),
]:
    print(
        f'--- {label} run (pk={run.pk}, source={run.source!r}, '
        f'telescope_class={run.telescope_class!r}, site={run.site!r}) ---'
    )
    events = list(owned_events(run).order_by('start_time')) + list(allocation_events(run).order_by('start_time'))
    night_urls = [ev.url for ev in events if ALLOC_NIGHT_RE.match(ev.url)]
    if night_urls:
        per_night_run_pks.add(run.pk)
    else:
        container_only_run_pks.add(run.pk)
    for ev in events:
        print(f'  url={ev.url!r}')
        print(f'    title={ev.title!r}')
        print(f'    telescope={ev.telescope!r}  instrument={ev.instrument!r}')
        print(f'    start={ev.start_time.isoformat()}  end={ev.end_time.isoformat()}')
    print()

print(f'Per-night runs (ALLOC:{{pk}}:{{date}}): {sorted(per_night_run_pks)}')
print(f'Container-only runs (bare RUN:{{pk}}): {sorted(container_only_run_pks)}')

assert per_night_run_pks == {classical_run.pk}, (
    'Only the classical run (non-queue source, resolved ground site, no telescope_class) '
    'dispatches to the per-night ALLOC: family -- dispatches_per_night() is the single '
    'decision point.'
)
assert container_only_run_pks == {lco_queue_run.pk, eso_queue_run.pk, class_wide_run.pk}, (
    'The LCO-queue run, the ESO-queue run and the class-wide run all take the whole-window '
    'RUN: container -- queue-sourced and class-wide runs share the same branch.'
)

--- Classical run (pk=69, source='classical_file', telescope_class='', site=<Observatory: Y21: Y21 Classical Site (Campaign Lifecycle Demo)>) ---
  url='ALLOC:69:2026-09-01'
    title='Y21 0.4m SBIG-STX16803'
    telescope='Y21 0.4m'  instrument='SBIG-STX16803'
    start=2026-09-01T22:34:40+00:00  end=2026-09-02T10:50:54+00:00
  url='ALLOC:69:2026-09-02'
    title='Y21 0.4m SBIG-STX16803'
    telescope='Y21 0.4m'  instrument='SBIG-STX16803'
    start=2026-09-02T22:35:12+00:00  end=2026-09-03T10:49:43+00:00
  url='ALLOC:69:2026-09-03'
    title='Y21 0.4m SBIG-STX16803'
    telescope='Y21 0.4m'  instrument='SBIG-STX16803'
    start=2026-09-03T22:35:44+00:00  end=2026-09-04T10:48:32+00:00

--- LCO queue run (pk=70, source='lco_queue', telescope_class='', site=<Observatory: Y22: Y22 LCO Network Site (Campaign Lifecycle Demo)>) ---
  url='RUN:70'
    title='Y22 1m0-SciCam-Sinistro (window 2026-09-05..2026-09-07)'
    telescope='Y22 1m0-SciCam-Sinistro'  instrument=''
    start=2026-09-05T00

## The public campaign table

`public_client.get(reverse('campaigns:table', args=[campaign.pk]))` -- the SAME anonymous
client used for the submission form, never logged in. This is the public read path
(VIEW-01/04): soft-filtered via `.values()` so pending rows are excluded from the SQL
SELECT entirely (in contrast to the two staff queues above, both hard-gated by
`StaffRequiredMixin`), and contact PII is shown only for rows whose submitter opted in --
which none of the demo runs did. Contact fields are never printed into this notebook's
committed output even when they are empty (33-05 P1 / 33-08 P5) -- the cell below prints
only `pk`, `telescope_instrument` and `approval_status` for each row.

The rejected demo run (5th) is visible here too, alongside the four lifecycle runs:
`CampaignRunTableView` excludes only `pending_review` rows from the public table, so
rejection removes a run from attribution candidacy (above) but NOT from the public table --
those are two independent facts about a run, and the public table's job is never to answer
"was this offered as an attribution match."


In [20]:
response = public_client.get(reverse('campaigns:table', args=[campaign.pk]))
assert response.status_code == 200

table = response.context['table']
print(f'Public table rows for {campaign.name!r} (campaign pk={campaign.pk}):')
# Contact fields are never printed into committed notebook output (33-05 P1 / 33-08 P5)
# even when they are empty, so a future edit does not reintroduce them.
for row in table.page.object_list:
    record = row.record
    print(
        f'  pk={record["pk"]}  telescope_instrument={record["telescope_instrument"]!r}  '
        f'approval_status={record["approval_status"]!r}'
    )

visible_pks = {row.record['pk'] for row in table.page.object_list}
# The rejected demo run (5th) is included: CampaignRunTableView excludes only
# pending_review, so a REJECTED run is still publicly listed -- rejection removes a run
# from attribution candidacy (27-REVIEW IN-02), not from the public table.
assert visible_pks == {
    classical_run.pk,
    lco_queue_run.pk,
    eso_queue_run.pk,
    class_wide_run.pk,
    rejected_demo_run.pk,
}

Public table rows for 'Campaign Lifecycle Demo' (campaign pk=13):
  pk=71  telescope_instrument='UT1/FORS2'  approval_status='approved'
  pk=70  telescope_instrument='Y22 1m0-SciCam-Sinistro'  approval_status='approved'
  pk=69  telescope_instrument='Y21 0.4m/SBIG-STX16803'  approval_status='approved'
  pk=72  telescope_instrument='LCO 1m0 Network (Campaign Lifecycle Demo)'  approval_status='approved'
  pk=74  telescope_instrument='Y21 0.4m/SBIG-STX16803'  approval_status='rejected'


## The public run tally and campaign roll-up (Phase 37 TALLY-01/TALLY-02)

Every row on this SAME public table response already carries a "Progress" cell --
group/record counts plus the four ordered night segments (`[O]`/`[S]`/`[X/F]`/`[U]`) --
computed once for the whole table by `campaign_tally.tallies_for_runs()`, never a
per-row query (D-08). The roll-up strip above the table sums the same segments across
the campaign's approved runs (D-10). A second GET with `staff_client` proves the
anonymous and the staff response render identical segments for the same run -- this is
a public tally, not a staff-only one (the "public, not staff-only" half of TALLY-01).

In [21]:
content = response.content.decode()


def _segment_text(segment):
    if not segment['known']:
        return f"{segment['marker']} not yet known"
    if segment['is_estimate']:
        return f"{segment['marker']} \u2248{segment['count']}"
    return f"{segment['marker']} {segment['count']}"


response_staff = staff_client.get(reverse('campaigns:table', args=[campaign.pk]))
assert response_staff.status_code == 200
content_staff = response_staff.content.decode()

for label, run in [
    ('Classical', classical_run),
    ('LCO queue', lco_queue_run),
    ('ESO queue', eso_queue_run),
    ('Class-wide', class_wide_run),
]:
    tally = campaign_tally.get_or_compute_tally(run)
    segments = campaign_tally.tally_segments(tally)
    segment_bits = [_segment_text(s) for s in segments]
    print(
        f'{label:<12} pk={run.pk}  {tally["groups"]} group(s) \u00b7 {tally["records"]} record(s) \u00b7 '
        + ' '.join(segment_bits)
    )
    for bit in segment_bits:
        assert bit in content, f'{bit!r} missing from the public table for {label}'
        assert bit in content_staff, f'{bit!r} missing from the staff table for {label}'

print()
print(
    'Anonymous and staff responses render identical Progress-cell segments for every run -- '
    'this tally is public, not staff-only (TALLY-01).'
)

# The campaign roll-up strip above the table sums the same segments across the campaign's
# approved, publicly visible runs (TALLY-02).
rollup = campaign_tally.campaign_rollup(campaign)
assert f"{rollup['groups']} group" in content, 'roll-up group count missing from the page'
assert f"{rollup['records']} record" in content, 'roll-up record count missing from the page'
print(
    f"Campaign roll-up: {rollup['groups']} group(s) \u00b7 {rollup['records']} record(s) over "
    f"{rollup['runs']} run(s)"
)
for segment in campaign_tally.tally_segments(rollup):
    print(f"  {segment['label']}: {_segment_text(segment)}")
assert (
    rollup['nights_unused'] is not None and rollup['nights_unused'] >= 3
), "the roll-up must fold in classical_run's own exact 3 unused nights"
assert rollup['unused_is_estimate'] is False, 'no run here carries a fetched proposal estimate yet'

Classical    pk=69  0 group(s) · 0 record(s) · [O] 0 [S] 0 [X/F] 0 [U] 3
LCO queue    pk=70  0 group(s) · 0 record(s) · [O] 0 [S] 0 [X/F] 0 [U] not yet known
ESO queue    pk=71  0 group(s) · 0 record(s) · [O] 0 [S] 0 [X/F] 0 [U] not yet known
Class-wide   pk=72  0 group(s) · 0 record(s) · [O] 0 [S] 0 [X/F] 0 [U] not yet known

Anonymous and staff responses render identical Progress-cell segments for every run -- this tally is public, not staff-only (TALLY-01).
Campaign roll-up: 0 group(s) · 0 record(s) over 5 run(s)
  Observed: [O] 0
  Scheduled: [S] 0
  Expired/failed: [X/F] 0
  Unused awarded night: [U] 3


## An unused awarded night (Phase 37 UNUSED-01)

`classical_run`'s three `ALLOC:` nights (2026-09-01..2026-09-03, seeded at submission
time above) already demonstrate the ordinary case: the window elapsed with nothing
scheduled or observed on it, so every night reads unused -- the same exact count the
tally above just showed. For the D-14 precedence rule (a staff-set run status always
wins over "unused"), seed one more small run with an elapsed single-night `ALLOC:`
window at the same site, marked `weather_tech_failure`: its own elapsed night must show
the `[W]` marker, never the muted `[U]` chip, even though the underlying elapsed-with-
nothing-observed fact is identical. Render the month view with the public client and
check both runs' nights side by side; then confirm the render never rewrote either
run's stored `CalendarEvent` titles (D-12's no-write contract).

In [22]:
from datetime import date as date_cls

from solsys_code.campaign_reconciler import reconcile_run

weathered_run, _ = CampaignRun.objects.update_or_create(
    campaign=None,
    telescope_instrument='Y21 0.4m/SBIG-STX16803 (weathered, unused-night demo)',
    window_start=date_cls(2026, 9, 1),
    window_end=date_cls(2026, 9, 1),
    defaults=dict(
        site=y21_site,
        site_raw='Y21',
        approval_status=CampaignRun.ApprovalStatus.APPROVED,
        source=CampaignRun.Source.CLASSICAL_FILE,
        run_status=CampaignRun.RunStatus.WEATHER_TECH_FAILURE,
    ),
)
reconcile_run(weathered_run)

classical_titles_before = list(
    CalendarEvent.objects.filter(url__startswith=f'ALLOC:{classical_run.pk}:')
    .order_by('url')
    .values_list('title', flat=True)
)
weathered_titles_before = list(
    CalendarEvent.objects.filter(url__startswith=f'ALLOC:{weathered_run.pk}:')
    .order_by('url')
    .values_list('title', flat=True)
)
print(f'classical_run titles before render: {classical_titles_before}')
print(f'weathered_run titles before render: {weathered_titles_before}')
assert weathered_titles_before and all(t.startswith('[W]') for t in weathered_titles_before)

month_response = public_client.get(reverse('calendar:calendar'), {'year': 2026, 'month': 9})
assert month_response.status_code == 200
month_content = month_response.content.decode()

assert (
    'cal-event-unused' in month_content
), "classical_run's elapsed, ordinary-status nights must render the unused chip class"
assert 'data-unused="1"' in month_content
unused_token_count = month_content.count('[U] ')
print(f'Occurrences of the [U] token in the rendered month view: {unused_token_count}')
assert unused_token_count >= 3, "all three of classical_run's elapsed nights should carry the [U] token"

# D-14: a staff-set run status always wins -- the weathered run's own elapsed night must
# show [W], never the muted [U] chip, even though it is equally elapsed with nothing
# scheduled or observed on it.
assert '[W] Y21' in month_content

classical_titles_after = list(
    CalendarEvent.objects.filter(url__startswith=f'ALLOC:{classical_run.pk}:')
    .order_by('url')
    .values_list('title', flat=True)
)
weathered_titles_after = list(
    CalendarEvent.objects.filter(url__startswith=f'ALLOC:{weathered_run.pk}:')
    .order_by('url')
    .values_list('title', flat=True)
)
assert (
    classical_titles_after == classical_titles_before
), 'rendering the month view must never rewrite a stored title (D-12)'
assert weathered_titles_after == weathered_titles_before

print()
print('classical_run (ordinary status, elapsed, nothing observed): shows the muted [U] chip.')
print('weathered_run (staff-marked weathered, elapsed, nothing observed): shows [W], never [U] -- D-14.')
print('Stored CalendarEvent titles are byte-identical before and after rendering (D-12 no-write contract).')

classical_run titles before render: ['Y21 0.4m SBIG-STX16803', 'Y21 0.4m SBIG-STX16803', 'Y21 0.4m SBIG-STX16803']
weathered_run titles before render: ['[W] Y21 0.4m SBIG-STX16803 (weathered, unused-night demo)']
Occurrences of the [U] token in the rendered month view: 9

classical_run (ordinary status, elapsed, nothing observed): shows the muted [U] chip.
weathered_run (staff-marked weathered, elapsed, nothing observed): shows [W], never [U] -- D-14.
Stored CalendarEvent titles are byte-identical before and after rendering (D-12 no-write contract).


## Summary

Across this notebook, one campaign and four `CampaignRun`s went from public submission
(`campaigns:submit`, anonymous) to fully on the calendar: staff approved all four through
the approval queue (`campaigns:decide` with `action=approve`); the one run whose free-text
site didn't resolve (the ESO run) surfaced in Sites Needing Review and was resolved
through a second, explicit staff action (`action=resolve_site`); one pre-existing,
hand-entered calendar event was attributed to the classical run through the attribution
queue (`campaigns:attribution_decide` with `action=confirm`); and the reconciler -- invoked
automatically by every one of those staff actions, never run as a separate batch command
here -- projected all four runs' `CalendarEvent` rows. Only the classical run (a
non-queue, non-class-wide run with a resolved ground site) dispatched to the per-night
`ALLOC:{pk}:{night}` family, one sunset-to-sunrise event per observing night; the LCO-queue
run, the ESO-queue run and the class-wide run all dispatched to the same whole-window
`RUN:{pk}` container instead -- `dispatches_per_night()` decides on source/site/class, not
on whether a run carries a real ground site.

A fifth `CampaignRun` (the rejected demo run) also appeared, deliberately outside this
four-run lifecycle: submitted through the same public form and then rejected through the
same staff decision endpoint, it demonstrated that a rejected run is never offered as an
attribution match (27-REVIEW IN-02) while remaining visible on the public campaign table --
rejection and attribution-candidacy are independent facts about a run.

See `docs/runbooks/telescope_runs_calendar.rst`, "How do I get every campaign run onto the
calendar?", for the operator-facing version of the window-shape rule demonstrated above,
and its "How do I attribute existing calendar events and observation records to a run?"
section for the rejected-run exclusion the fifth run demonstrates; and
`reconcile_campaign_runs_demo.ipynb` for the reconciler command itself (`--dry-run`
counters, idempotency, sweeps and backfills) -- deliberately not duplicated here.


Beyond the four-run lifecycle above, this notebook also demonstrates the campaign-annotation
story added in Phase 33: the confirmed orphan event's modal renders an "Attributed campaign
run" block and the month calendar renders a `.cal-campaign-chip` marker, both sourced from
`CalendarEventMeta.run` at request time rather than from anything written into the event
(ROADMAP criterion 3 -- the decoration survives a from-scratch rewrite of the event's own
title and description); and `unlink_event_from_run()` clears that attribution -- and only that
attribution -- leaving the `CalendarEvent` and the `CalendarEventMeta` row otherwise untouched
and deleting nothing (ROADMAP criterion 4). A final cell shows the two new series-identity link
fields, `observation_record` and `observation_group`, which no code writes yet in this phase --
Phase 34's observation projector is the one that will -- and confirms that an event carrying
only those links (no attribution) is still offered by the event attribution queue exactly like
any other unattributed event (D-15).


## G-37-5: the Progress tally follows the rendered page, not a guess at it (Phase 37 gap closure)

The campaign this notebook has been building has five runs, all of which fit on the
table's single page -- a sorted or per-page-widened request cannot show the difference
between "the tally covers the rendered rows" and "the tally covers a slice computed
before the request's own sort/page/per_page parameters were applied." That defect
(G-37-5, code review finding CR-01) only appears once a campaign has more than one page
of runs: before this fix, `CampaignRunTableView.get_table_kwargs()` reimplemented
django-tables2's page resolution -- reading `page`, hardcoding `per_page` at 25, and
slicing the queryset in `window_start` order -- BEFORE `RequestConfig.configure()` had
applied the request's `sort` and `per_page` parameters. A sorted or per-page-widened
request then rendered a different set of rows than the ones the tally was computed for,
and those uncovered rows fell back to a muted "Progress not available" cell.

The fix moves the tally fetch to AFTER `RequestConfig.configure()` has resolved the
final page: `CampaignRunTableView.get_table()` now reads `table.paginated_rows` -- the
exact `BoundRows` the django-tables2 table template iterates -- so the tallies dict
covers exactly the rendered rows for every combination of `sort`, `page` and `per_page`.
Because the incoming `per_page` is otherwise unbounded once the tally follows the
rendered rows, a request-side cap (`campaign_views.MAX_TABLE_PER_PAGE`) keeps an
anonymous `?per_page=` from fanning the per-run tally pass out across an entire campaign.

The cell below builds its own separate 30-run campaign -- large enough to span more than
one page at the table's 25-row default -- and shows a plain request, a
`?sort=-telescope_instrument` request and a `?per_page=50` request all rendering zero
"Progress not available" cells.

In [23]:
from datetime import timedelta

from solsys_code import campaign_views

TALLY_DEMO_CAMPAIGN_NAME = 'G-37-5 Tally Coverage Demo Campaign'
TALLY_DEMO_RUN_COUNT = 30

tally_demo_campaign, _ = TargetList.objects.get_or_create(name=TALLY_DEMO_CAMPAIGN_NAME)
# Idempotent re-run: delete this demo's own prior runs before recreating them, mirroring
# the notebook's own "Demo-scoped reset" pattern above -- never a blanket
# CampaignRun.objects.all().delete().
deleted_tally_demo_runs, _ = CampaignRun.objects.filter(campaign=tally_demo_campaign).delete()
if deleted_tally_demo_runs:
    print(f'Reset {TALLY_DEMO_CAMPAIGN_NAME!r}: deleted {deleted_tally_demo_runs} prior demo run(s).')

tally_demo_base_date = date_cls(2026, 1, 1)
tally_demo_runs = []
for i in range(TALLY_DEMO_RUN_COUNT):
    # Descending window_start as i increases, ascending telescope_instrument label as i
    # increases -- the two orderings are exact opposites of each other, so sorting by
    # telescope_instrument genuinely re-orders the page against get_queryset()'s default
    # window_start-descending order.
    window_date = tally_demo_base_date - timedelta(days=i)
    tally_demo_runs.append(
        CampaignRun.objects.create(
            campaign=tally_demo_campaign,
            approval_status=CampaignRun.ApprovalStatus.APPROVED,
            telescope_instrument=f'Y21/G375-{i:02d}',
            site=y21_site,
            site_raw=y21_site.obscode,
            window_start=window_date,
            window_end=window_date,
        )
    )
print(
    f'Created {len(tally_demo_runs)} CampaignRun rows on {tally_demo_campaign.name!r} '
    f'(pk={tally_demo_campaign.pk}).'
)

tally_demo_url = reverse('campaigns:table', args=[tally_demo_campaign.pk])
tally_demo_requests = [
    ('plain', {}),
    ('sorted (?sort=-telescope_instrument)', {'sort': '-telescope_instrument'}),
    ('widened (?per_page=50)', {'per_page': '50'}),
]

print()
tally_demo_rendered_counts = {}
for label, params in tally_demo_requests:
    response = public_client.get(tally_demo_url, params)
    assert response.status_code == 200
    content = response.content.decode()
    not_available_count = content.count('Progress not available')
    rendered_count = len(list(response.context['table'].paginated_rows))
    tally_demo_rendered_counts[label] = rendered_count
    print(f'{label:<38} rendered_rows={rendered_count:<3} not_available_count={not_available_count}')
    assert not_available_count == 0, f'{label} rendered a row with no Progress tally'

assert tally_demo_rendered_counts['plain'] == 25, 'the plain request should render one 25-row page'
assert (
    tally_demo_rendered_counts['sorted (?sort=-telescope_instrument)'] == 25
), 'the sorted request should still render one 25-row page'
assert (
    tally_demo_rendered_counts['widened (?per_page=50)'] == 30
), 'per_page=50 is under the MAX_TABLE_PER_PAGE cap and must render all 30 rows'

print()
print(
    f'All three requests render zero "Progress not available" cells -- the tally now covers '
    f'exactly the rows django-tables2 resolves for sort/page/per_page, not a pre-RequestConfig '
    f'guess at them. An unbounded per_page is separately capped at '
    f'campaign_views.MAX_TABLE_PER_PAGE={campaign_views.MAX_TABLE_PER_PAGE} so a public GET '
    f'cannot fan the tally pass out across an entire campaign.'
)

Created 30 CampaignRun rows on 'G-37-5 Tally Coverage Demo Campaign' (pk=15).

plain                                  rendered_rows=25  not_available_count=0
sorted (?sort=-telescope_instrument)   rendered_rows=25  not_available_count=0
widened (?per_page=50)                 rendered_rows=30  not_available_count=0

All three requests render zero "Progress not available" cells -- the tally now covers exactly the rows django-tables2 resolves for sort/page/per_page, not a pre-RequestConfig guess at them. An unbounded per_page is separately capped at campaign_views.MAX_TABLE_PER_PAGE=100 so a public GET cannot fan the tally pass out across an entire campaign.


## Scratch database teardown

Removes the scratch copy created in the setup cell above. The developer database
was never opened for writing by this notebook run.

In [24]:
import shutil

shutil.rmtree(scratch_db_dir, ignore_errors=True)
print(f'Removed scratch database directory: {scratch_db_dir}')
print('The developer database (src/fomo_db.sqlite3) was never opened for writing.')

Removed scratch database directory: /tmp/fomo-notebook-db-p479d9hg
The developer database (src/fomo_db.sqlite3) was never opened for writing.
